In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.106 Safari/537.36'}

main_page_1 = 'https://www.transfermarkt.com/wettbewerbe/'
main_page_2 = '/wettbewerbe?ajax=yw1&plus=1&page='
lands = ['europa', 'amerika', 'afrika', 'asien']
lands_while_end = {'europa': 'Third Tier',
                   'amerika': 'Third Tier',
                   'afrika': 'Domestic Cup',
                   'asien': 'Third Tier'}

In [2]:
# deklaracja tablic do stworzenia data setu o zawodnikach
league_names = []
leagues_tier = []
countries = []
clubs_nos = []
players_nos = []
foreigners_percentages = []
gpms = []
total_values = []

In [3]:
def get_value(value):
    int_value = value[1:-1]
    multipler = value [-1]
    if multipler == 'n':
        return int(float(value[1:-2])*1000000000)
    elif multipler == 'm':
        return int(float(int_value)*1000000)
    else:
        return int(float(int_value)*1000)

In [4]:
for continent in lands:
    loop_end = lands_while_end[continent]
    actual_tier = ""
    for page_number in range (1,7):
        url = main_page_1 +  continent + main_page_2 + str(page_number)
        pageTree = requests.get(url, headers=headers)
        pageSoup = BeautifulSoup(pageTree.content, 'lxml')
        leagues = pageSoup.find("table", {"class": "items"}).find("tbody").find_all("tr")
        for league in leagues:
            league_data = league.find_all("td")

            if len(league_data) == 1:
                actual_tier = league_data[0].contents[0].strip()

            elif len(league_data)> 2:
                league_name = league_data[2].find("a").contents[0]
                country = league_data[3].find("img")['title']
                clubs_no = league_data[4].contents[0]
                players_no = league_data[5].contents[0]
                f_p = league_data[7].find("a").contents[0]
                if continent == 'europa':
                    gpm = league_data[9].find("a").contents[0]
                else:
                    gpm = league_data[8].find("a").contents[0] if league_data[8].find("a") else None
                
                value = league_data[-1].contents[0]

                league_names.append(league_name)
                leagues_tier.append(actual_tier)
                countries.append(country)
                clubs_nos.append(clubs_no)
                players_nos.append(players_no)
                foreigners_percentages.append(round(float(f_p[:-2])/100*float(players_no)))
                gpms.append(gpm)
                total_values.append(get_value(value) if value != '-' else None)

            if actual_tier == loop_end:
                break
        
        if actual_tier == loop_end:
            break


In [5]:
# budujemy data frame'a
df = pd.DataFrame({
    "League_name":league_names,
    "League_tier":leagues_tier,
    "Country":countries,
    "Club_no":clubs_nos, 
    "Players_no":players_nos, 
    "Foreigners": foreigners_percentages, 
    "Goals_per_match": gpms, 
    "Total_value": total_values, 
    })

In [6]:
df

,League_name,League_tier,Country,Club_no,Players_no,Foreigners,Goals_per_match,Total_value
0,Premier League,First Tier,England,20,544,388,2.83,1.253000e+10
1,LaLiga,First Tier,Spain,20,501,216,2.56,5.450000e+09
2,Serie A,First Tier,Italy,20,572,383,2.34,5.320000e+09
3,Bundesliga,First Tier,Germany,18,524,307,3.16,4.810000e+09
4,Ligue 1,First Tier,France,18,478,299,2.85,3.790000e+09
...,...,...,...,...,...,...,...,...
213,NPL New South Wales,Second Tier,Australia,16,567,73,2.98,2.400000e+06
214,NPL Victoria,Second Tier,Australia,14,438,75,3.30,1.100000e+06
215,Premier League 2,Second Tier,Singapore,8,152,14,3.72,2.550000e+05
216,Football League Div.2,Second Tier,Chinese Taipei,7,126,13,3.48,2.500000e+04


In [7]:
nan_rows = df.isna().any(axis=1)
df = df[~nan_rows]

In [8]:
df['Country'] = df['Country'].astype(str)
df['Club_no'] = df['Club_no'].astype(int)
df['Players_no'] = df['Players_no'].astype(str).str.replace(".","").astype(int)
df['Foreigners'] = df['Foreigners'].astype(int)
df['Goals_per_match'] = df['Goals_per_match'].astype(float)
df['Total_value'] = df['Total_value'].astype(int)

C:\Users\flork\AppData\Local\Temp\ipykernel_868\3795448596.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Country'] = df['Country'].astype(str)
C:\Users\flork\AppData\Local\Temp\ipykernel_868\3795448596.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Club_no'] = df['Club_no'].astype(int)
C:\Users\flork\AppData\Local\Temp\ipykernel_868\3795448596.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

In [9]:
df.dtypes

League_name         object
League_tier         object
Country             object
Club_no              int64
Players_no           int64
Foreigners           int64
Goals_per_match    float64
Total_value          int64
dtype: object

In [10]:
df

,League_name,League_tier,Country,Club_no,Players_no,Foreigners,Goals_per_match,Total_value
0,Premier League,First Tier,England,20,544,388,2.83,12530000000
1,LaLiga,First Tier,Spain,20,501,216,2.56,5450000000
2,Serie A,First Tier,Italy,20,572,383,2.34,5320000000
3,Bundesliga,First Tier,Germany,18,524,307,3.16,4810000000
4,Ligue 1,First Tier,France,18,478,299,2.85,3790000000
...,...,...,...,...,...,...,...,...
213,NPL New South Wales,Second Tier,Australia,16,567,73,2.98,2400000
214,NPL Victoria,Second Tier,Australia,14,438,75,3.30,1100000
215,Premier League 2,Second Tier,Singapore,8,152,14,3.72,255000
216,Football League Div.2,Second Tier,Chinese Taipei,7,126,13,3.48,25000


In [11]:
group_cols = [
    "Country",
    "League_tier",
    "Club_no",
    "Players_no",
    "Foreigners",
    "Total_value"
]

fixed_df = (
    df
    .groupby(group_cols, as_index=False)
    .agg({
        "Goals_per_match": "mean",
        "League_name": "first"
    })
)

fixed_df

,Country,League_tier,Club_no,Players_no,Foreigners,Total_value,Goals_per_match,League_name
0,Albania,First Tier,10,268,131,48410000,2.02,Kategoria Superiore
1,Albania,Second Tier,12,299,109,14540000,2.25,Kategoria e Parë
2,Algeria,First Tier,16,427,43,96900000,2.06,Ligue 1
3,Andorra,First Tier,10,214,188,26170000,2.85,Primera Divisió
4,Andorra,Second Tier,8,246,99,1310000,4.92,Segona Divisió
...,...,...,...,...,...,...,...,...
193,Venezuela,Second Tier,16,464,46,31560000,2.39,Liga FUTVE 2
194,Vietnam,First Tier,14,440,85,57550000,2.67,V.League 1
195,Wales,First Tier,12,275,97,14180000,3.16,Cymru Premier
196,Wales,Second Tier,16,374,53,25000,2.87,Cymru South


In [12]:
new_order = [
    "Country",
    "League_name",
    "League_tier",
    "Club_no",
    "Players_no",
    "Foreigners",
    "Goals_per_match",
    "Total_value"
]

fixed_df = fixed_df[new_order]

In [13]:
fixed_df

,Country,League_name,League_tier,Club_no,Players_no,Foreigners,Goals_per_match,Total_value
0,Albania,Kategoria Superiore,First Tier,10,268,131,2.02,48410000
1,Albania,Kategoria e Parë,Second Tier,12,299,109,2.25,14540000
2,Algeria,Ligue 1,First Tier,16,427,43,2.06,96900000
3,Andorra,Primera Divisió,First Tier,10,214,188,2.85,26170000
4,Andorra,Segona Divisió,Second Tier,8,246,99,4.92,1310000
...,...,...,...,...,...,...,...,...
193,Venezuela,Liga FUTVE 2,Second Tier,16,464,46,2.39,31560000
194,Vietnam,V.League 1,First Tier,14,440,85,2.67,57550000
195,Wales,Cymru Premier,First Tier,12,275,97,3.16,14180000
196,Wales,Cymru South,Second Tier,16,374,53,2.87,25000


In [14]:
remove_leagues = ['Liga 1 Clausura', 'Liga Panameña Apertura', 'National League - Central', 'National League - South', 'National League - North', 'Liguilla Apertura', 'Liguilla de Expansión Apertura', 'Apertura Play-Off']
fixed_df = fixed_df[~fixed_df["League_name"].isin(remove_leagues)].reset_index()
fixed_df

,index,Country,League_name,League_tier,Club_no,Players_no,Foreigners,Goals_per_match,Total_value
0,0,Albania,Kategoria Superiore,First Tier,10,268,131,2.02,48410000
1,1,Albania,Kategoria e Parë,Second Tier,12,299,109,2.25,14540000
2,2,Algeria,Ligue 1,First Tier,16,427,43,2.06,96900000
3,3,Andorra,Primera Divisió,First Tier,10,214,188,2.85,26170000
4,4,Andorra,Segona Divisió,Second Tier,8,246,99,4.92,1310000
...,...,...,...,...,...,...,...,...,...
185,193,Venezuela,Liga FUTVE 2,Second Tier,16,464,46,2.39,31560000
186,194,Vietnam,V.League 1,First Tier,14,440,85,2.67,57550000
187,195,Wales,Cymru Premier,First Tier,12,275,97,3.16,14180000
188,196,Wales,Cymru South,Second Tier,16,374,53,2.87,25000


In [15]:
group_cols = [
    "Country",
    "League_tier"
]

fixed_df_2 = (
    fixed_df
    .groupby(group_cols, as_index=False)
    .agg({
        "Goals_per_match": "mean",
        "Club_no": "sum",
        "Players_no": "sum",
        "Foreigners": "sum",
        "Total_value": "sum",
        "League_name": "first"
    })
)

fixed_df_2

,Country,League_tier,Goals_per_match,Club_no,Players_no,Foreigners,Total_value,League_name
0,Albania,First Tier,2.020,10,268,131,48410000,Kategoria Superiore
1,Albania,Second Tier,2.250,12,299,109,14540000,Kategoria e Parë
2,Algeria,First Tier,2.060,16,427,43,96900000,Ligue 1
3,Andorra,First Tier,2.850,10,214,188,26170000,Primera Divisió
4,Andorra,Second Tier,4.920,8,246,99,1310000,Segona Divisió
...,...,...,...,...,...,...,...,...
182,Venezuela,First Tier,2.335,14,406,80,82370000,Liga FUTVE Apertura
183,Venezuela,Second Tier,2.390,16,464,46,31560000,Liga FUTVE 2
184,Vietnam,First Tier,2.670,14,440,85,57550000,V.League 1
185,Wales,First Tier,3.160,12,275,97,14180000,Cymru Premier


In [16]:
new_order = [
    "Country",
    "League_name",
    "League_tier",
    "Club_no",
    "Players_no",
    "Foreigners",
    "Goals_per_match",
    "Total_value"
]

fixed_df_2 = fixed_df_2[new_order]

fixed_df_2

,Country,League_name,League_tier,Club_no,Players_no,Foreigners,Goals_per_match,Total_value
0,Albania,Kategoria Superiore,First Tier,10,268,131,2.020,48410000
1,Albania,Kategoria e Parë,Second Tier,12,299,109,2.250,14540000
2,Algeria,Ligue 1,First Tier,16,427,43,2.060,96900000
3,Andorra,Primera Divisió,First Tier,10,214,188,2.850,26170000
4,Andorra,Segona Divisió,Second Tier,8,246,99,4.920,1310000
...,...,...,...,...,...,...,...,...
182,Venezuela,Liga FUTVE Apertura,First Tier,14,406,80,2.335,82370000
183,Venezuela,Liga FUTVE 2,Second Tier,16,464,46,2.390,31560000
184,Vietnam,V.League 1,First Tier,14,440,85,2.670,57550000
185,Wales,Cymru Premier,First Tier,12,275,97,3.160,14180000


In [17]:
fixed_df_2.to_csv('leagues.csv', index=True)  